# XGBoost — Fair 24-Hour Forecast Comparison with Prophet

This notebook combines the previous XGBoost feature engineering, tuning, evaluation, and artifact saving into **one notebook**, while correcting the comparison issues.

### Corrections made
- Uses the **same four development folds as Prophet**:
  - August 2025
  - November 2025
  - February 2026
  - May 2026
- Uses **mean CV RMSE** as the primary model-selection metric, matching Prophet.
- Removes `lag_1 ... lag_23` and rolling-demand features that are not all known when issuing a complete next-24-hour forecast from one origin.
- Keeps only safe demand-history features:
  - `demand_lag_24`
  - `demand_lag_168`
- Keeps June 2026 completely untouched during feature selection/tuning/comparison.
- Uses forward-fill only for sparse exogenous values.
- Auto-detects `master_training_data.csv` under Kaggle `/kaggle/input`.
- Saves fold-level checkpoints and all comparison artifacts.
- Includes the already-completed best Prophet CV result so the XGBoost result can be compared immediately.

> **Run this notebook top-to-bottom.**  
> Keep `RUN_FINAL_JUNE_TEST = False` until the CV comparison selects the overall winner.


In [1]:
from pathlib import Path
from itertools import product
import json
import time

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

OUTPUT_FOLDER = (
    Path("/kaggle/working/xgboost_outputs")
    if Path("/kaggle/working").exists()
    else Path("artifacts/xgboost_fair_comparison")
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("XGBoost version:", xgb.__version__)
print("Output folder:", OUTPUT_FOLDER)


XGBoost version: 3.2.0
Output folder: /kaggle/working/xgboost_outputs


## 1. Find and load the master dataset

In [2]:
# Auto-detect the dataset so the notebook does not depend on Kaggle's exact mount path.
if Path("/kaggle/input").exists():
    matches = list(Path("/kaggle/input").rglob("master_training_data.csv"))
else:
    matches = list(Path(".").rglob("master_training_data.csv"))

if not matches:
    raise FileNotFoundError(
        "master_training_data.csv was not found. "
        "On Kaggle, use Add Input and attach the existing "
        "'master-train-data-uk-demand' dataset."
    )

# Prefer the known dataset folder if multiple copies exist.
preferred = [
    p for p in matches
    if "master-train-data-uk-demand" in str(p)
]
INPUT_PATH = preferred[0] if preferred else matches[0]

print("Using:", INPUT_PATH)

df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["timestamp"],
    low_memory=False,
)

df = (
    df
    .dropna(subset=["timestamp", "demand_mw"])
    .sort_values("timestamp")
    .drop_duplicates(subset=["timestamp"], keep="last")
    .reset_index(drop=True)
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())

# Row-based 24/168-hour lags are valid only if the dataset is truly hourly.
hour_diffs = df["timestamp"].diff().dropna()
assert (hour_diffs == pd.Timedelta(hours=1)).all(), (
    "Dataset is not perfectly hourly. "
    "Fix missing timestamps before using row-based demand lags."
)


Using: /kaggle/input/datasets/kusalnirukshan/master-train-data-uk-demand/master_training_data.csv
Rows: 144600
Columns: 51
Range: 2010-01-01 00:00:00 to 2026-06-30 23:00:00
Duplicate timestamps: 0


## 2. Isolate June 2026 before tuning

June is the final untouched holdout and is **not** used for XGBoost feature selection, hyperparameter tuning, or the Prophet-vs-XGBoost comparison.


In [3]:
FINAL_TEST_START = pd.Timestamp("2026-06-01 00:00:00")

# Safe historical-demand features for a rolling next-24-hour forecast.
df["demand_lag_24"] = df["demand_mw"].shift(24)
df["demand_lag_168"] = df["demand_mw"].shift(168)

# Time features known in advance.
df["year"] = df["timestamp"].dt.year
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["time_idx"] = np.arange(len(df), dtype=np.int64)

# Cyclical encodings: known for every future timestamp.
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

dev_df = df[df["timestamp"] < FINAL_TEST_START].copy()
final_test_df = df[df["timestamp"] >= FINAL_TEST_START].copy()

print(
    "Development:",
    len(dev_df),
    dev_df["timestamp"].min(),
    "to",
    dev_df["timestamp"].max(),
)
print(
    "FINAL TEST:",
    len(final_test_df),
    final_test_df["timestamp"].min(),
    "to",
    final_test_df["timestamp"].max(),
)

assert dev_df["timestamp"].max() < FINAL_TEST_START
assert final_test_df["timestamp"].min() == FINAL_TEST_START

print("\nJune 2026 remains LOCKED.")


Development: 143880 2010-01-01 00:00:00 to 2026-05-31 23:00:00
FINAL TEST: 720 2026-06-01 00:00:00 to 2026-06-30 23:00:00

June 2026 remains LOCKED.


## 3. Build 24-hour-safe XGBoost features

The previous pipeline created `lag_1 ... lag_24` and rolling statistics from immediately preceding demand. Those are useful for one-step-ahead prediction, but for a single next-24-hour forecast made at time **T**, features such as `lag_1` for **T+24** would require demand from **T+23**, which is not yet known.

This version therefore keeps only demand history that is available throughout the complete next-24-hour horizon: 24-hour and 168-hour lags.

Weather, calendar, economic lag variables, and timestamp-derived features are retained because they are exogenous or known/forecastable at prediction time.


In [4]:
# Curated feature groups.
weather_features = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
]

economic_features = [
    "econ_industrial_production_index_lag1m",
    "econ_gdp_index_lag1m",
    "econ_cpi_index_lag1m",
    "econ_unemployment_rate_lag1m",
    "econ_economic_data_complete",
]

calendar_features = [
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "weekend",
    "is_holiday",
    "cal_event_count",
    "cal_is_covid_lockdown",
    "cal_is_general_election",
    "cal_is_major_football",
    "cal_is_non_working_day",
    "year",
    "day_of_year",
    "time_idx",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

lag_features = [
    "demand_lag_24",
    "demand_lag_168",
]

feature_cols = (
    weather_features
    + economic_features
    + calendar_features
    + lag_features
)

# Keep only columns that actually exist.
feature_cols = [c for c in feature_cols if c in df.columns]

# Convert all selected regressors to numeric.
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Forward fill only: never back-fill time-series features.
df[feature_cols] = df[feature_cols].ffill()

# Re-split after cleaning the feature columns.
dev_df = df[df["timestamp"] < FINAL_TEST_START].copy()
final_test_df = df[df["timestamp"] >= FINAL_TEST_START].copy()

# 168-hour warm-up rows cannot have all required lags.
dev_model_df = dev_df.dropna(
    subset=feature_cols + ["demand_mw"]
).copy()

print("Number of XGBoost features:", len(feature_cols))
print(feature_cols)
print("\nDevelopment rows usable:", len(dev_model_df))

missing_dev = dev_model_df[feature_cols].isna().sum()
assert missing_dev.sum() == 0, "Unexpected missing feature values remain in development data."


Number of XGBoost features: 38
['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'precipitation', 'rain', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'econ_industrial_production_index_lag1m', 'econ_gdp_index_lag1m', 'econ_cpi_index_lag1m', 'econ_unemployment_rate_lag1m', 'econ_economic_data_complete', 'hour', 'day_of_week', 'day_of_month', 'month', 'weekend', 'is_holiday', 'cal_event_count', 'cal_is_covid_lockdown', 'cal_is_general_election', 'cal_is_major_football', 'cal_is_non_working_day', 'year', 'day_of_year', 'time_idx', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'demand_lag_24', 'demand_lag_168']

Development rows usable: 143712


## 4. Exact same four folds used by Prophet

In [5]:
cv_folds = [
    ("aug_2025", "2025-08-01 00:00:00", "2025-08-31 23:00:00"),
    ("nov_2025", "2025-11-01 00:00:00", "2025-11-30 23:00:00"),
    ("feb_2026", "2026-02-01 00:00:00", "2026-02-28 23:00:00"),
    ("may_2026", "2026-05-01 00:00:00", "2026-05-31 23:00:00"),
]

for name, start, end in cv_folds:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    train_rows = (dev_model_df["timestamp"] < start).sum()
    valid_rows = (
        (dev_model_df["timestamp"] >= start)
        & (dev_model_df["timestamp"] <= end)
    ).sum()

    print(
        f"{name}: train={train_rows:,}, valid={valid_rows:,}, "
        f"{start} -> {end}"
    )

    assert end < FINAL_TEST_START

print("\nAll XGBoost CV folds end before June 2026.")


aug_2025: train=136,416, valid=744, 2025-08-01 00:00:00 -> 2025-08-31 23:00:00
nov_2025: train=138,624, valid=720, 2025-11-01 00:00:00 -> 2025-11-30 23:00:00
feb_2026: train=140,832, valid=672, 2026-02-01 00:00:00 -> 2026-02-28 23:00:00
may_2026: train=142,968, valid=744, 2026-05-01 00:00:00 -> 2026-05-31 23:00:00

All XGBoost CV folds end before June 2026.


## 5. Metrics and one-fold evaluator

In [6]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    nonzero = np.abs(y_true) > 1e-8
    mape = (
        np.mean(
            np.abs(
                (y_true[nonzero] - y_pred[nonzero])
                / y_true[nonzero]
            )
        )
        * 100
    )

    r2 = r2_score(y_true, y_pred)

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "mape": float(mape),
        "r2": float(r2),
    }


def evaluate_xgb_fold(config_id, params, fold_name, valid_start, valid_end):
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    train = dev_model_df[
        dev_model_df["timestamp"] < valid_start
    ].copy()

    valid = dev_model_df[
        (dev_model_df["timestamp"] >= valid_start)
        & (dev_model_df["timestamp"] <= valid_end)
    ].copy()

    X_train = train[feature_cols]
    y_train = train["demand_mw"]

    X_valid = valid[feature_cols]
    y_valid = valid["demand_mw"]

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        eval_metric="rmse",
        **params,
    )

    print(
        f"\n{config_id} | {fold_name} | "
        f"train={len(train):,} valid={len(valid):,}"
    )

    t0 = time.time()
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False,
    )

    pred = model.predict(X_valid)
    metrics = calculate_metrics(y_valid, pred)

    print(
        {
            **{k: round(v, 4) for k, v in metrics.items()},
            "seconds": round(time.time() - t0, 2),
        }
    )

    return metrics


## 6. Small XGBoost hyperparameter search on the same four folds

The original search space is retained, but each sampled configuration is now evaluated on the **same four folds as Prophet**, and the winner is selected by **mean RMSE**.


In [7]:
param_dist = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6],
    "learning_rate": [0.03, 0.08],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [1, 2],
}

N_CONFIGS = 8

sampled_configs = list(
    ParameterSampler(
        param_dist,
        n_iter=N_CONFIGS,
        random_state=42,
    )
)

print("Sampled configurations:", len(sampled_configs))
for i, p in enumerate(sampled_configs, 1):
    print(i, p)


Sampled configurations: 8
1 {'subsample': 1.0, 'reg_lambda': 2, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 0.8}
2 {'subsample': 0.8, 'reg_lambda': 1, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 0.8}
3 {'subsample': 1.0, 'reg_lambda': 2, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
4 {'subsample': 1.0, 'reg_lambda': 2, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
5 {'subsample': 0.8, 'reg_lambda': 2, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 1.0}
6 {'subsample': 0.8, 'reg_lambda': 1, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 0.8}
7 {'subsample': 1.0, 'reg_lambda': 1, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 4, 

## 7. Run/resume XGBoost tuning with fold-level checkpoints

In [8]:
CHECKPOINT_PATH = OUTPUT_FOLDER / "xgb_tuning_folds_checkpoint.csv"

if CHECKPOINT_PATH.exists():
    tuning_fold_df = pd.read_csv(CHECKPOINT_PATH)
    print("Loaded checkpoint rows:", len(tuning_fold_df))
else:
    tuning_fold_df = pd.DataFrame()

completed_pairs = set()

if not tuning_fold_df.empty:
    completed_pairs = set(
        zip(
            tuning_fold_df["config_id"],
            tuning_fold_df["fold"],
        )
    )

total_expected = len(sampled_configs) * len(cv_folds)

print(
    f"Already completed: {len(completed_pairs)} / {total_expected}"
)

for config_number, params in enumerate(sampled_configs, start=1):
    config_id = f"xgb_config_{config_number:02d}"

    for fold_name, valid_start, valid_end in cv_folds:
        key = (config_id, fold_name)

        if key in completed_pairs:
            print("Skipping completed:", config_id, "|", fold_name)
            continue

        metrics = evaluate_xgb_fold(
            config_id=config_id,
            params=params,
            fold_name=fold_name,
            valid_start=valid_start,
            valid_end=valid_end,
        )

        new_row = {
            "config_id": config_id,
            "fold": fold_name,
            "params_json": json.dumps(params, sort_keys=True),
            **metrics,
        }

        tuning_fold_df = pd.concat(
            [tuning_fold_df, pd.DataFrame([new_row])],
            ignore_index=True,
        )

        completed_pairs.add(key)

        # Save after every completed fold.
        tuning_fold_df.to_csv(
            CHECKPOINT_PATH,
            index=False,
        )

        print(
            f"Checkpoint saved — "
            f"{len(completed_pairs)} / {total_expected} folds complete"
        )

print("\nXGBoost tuning loop finished.")


Already completed: 0 / 32

xgb_config_01 | aug_2025 | train=136,416 valid=744
{'mae': 684.1495, 'rmse': 927.5986, 'mape': 3.1857, 'r2': 0.9141, 'seconds': 4.02}
Checkpoint saved — 1 / 32 folds complete

xgb_config_01 | nov_2025 | train=138,624 valid=720
{'mae': 935.0688, 'rmse': 1217.3181, 'mape': 3.1685, 'r2': 0.9589, 'seconds': 3.79}
Checkpoint saved — 2 / 32 folds complete

xgb_config_01 | feb_2026 | train=140,832 valid=672
{'mae': 947.8716, 'rmse': 1295.1997, 'mape': 3.0842, 'r2': 0.9475, 'seconds': 3.89}
Checkpoint saved — 3 / 32 folds complete

xgb_config_01 | may_2026 | train=142,968 valid=744
{'mae': 772.1588, 'rmse': 1006.0945, 'mape': 3.517, 'r2': 0.907, 'seconds': 3.83}
Checkpoint saved — 4 / 32 folds complete

xgb_config_02 | aug_2025 | train=136,416 valid=744
{'mae': 793.5539, 'rmse': 1131.7013, 'mape': 3.7071, 'r2': 0.8722, 'seconds': 1.49}
Checkpoint saved — 5 / 32 folds complete

xgb_config_02 | nov_2025 | train=138,624 valid=720
{'mae': 1067.5285, 'rmse': 1361.5447, 'm

## 8. Summarize tuning and select the best XGBoost configuration

In [9]:
fold_counts = (
    tuning_fold_df
    .groupby("config_id")["fold"]
    .nunique()
)

complete_ids = fold_counts[
    fold_counts == len(cv_folds)
].index

complete_folds = tuning_fold_df[
    tuning_fold_df["config_id"].isin(complete_ids)
].copy()

tuning_summary = (
    complete_folds
    .groupby(["config_id", "params_json"], as_index=False)
    .agg(
        mean_mae=("mae", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_mape=("mape", "mean"),
        mean_r2=("r2", "mean"),
        std_rmse=("rmse", "std"),
        worst_fold_rmse=("rmse", "max"),
        min_fold_r2=("r2", "min"),
        folds_completed=("fold", "nunique"),
    )
    .sort_values(
        ["mean_rmse", "mean_mae"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

tuning_fold_df.to_csv(
    OUTPUT_FOLDER / "xgb_tuning_folds.csv",
    index=False,
)

tuning_summary.to_csv(
    OUTPUT_FOLDER / "xgb_tuning_summary.csv",
    index=False,
)

print(
    f"Complete configurations: "
    f"{len(tuning_summary)} / {len(sampled_configs)}"
)
display(tuning_summary)

assert len(tuning_summary) == len(sampled_configs), (
    "Not all XGBoost configurations completed."
)

best_xgb_row = tuning_summary.iloc[0]
best_xgb_params = json.loads(best_xgb_row["params_json"])

best_xgb_config = {
    "model_type": "XGBoost",
    "features": feature_cols,
    "params": best_xgb_params,
    "selection_metric": "mean_cv_rmse",
    "mean_cv_rmse": float(best_xgb_row["mean_rmse"]),
    "mean_cv_mae": float(best_xgb_row["mean_mae"]),
    "mean_cv_mape": float(best_xgb_row["mean_mape"]),
    "mean_cv_r2": float(best_xgb_row["mean_r2"]),
    "std_cv_rmse": float(best_xgb_row["std_rmse"]),
    "final_test_start": str(FINAL_TEST_START),
}

with open(
    OUTPUT_FOLDER / "best_xgb_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(best_xgb_config, f, indent=2)

print("\nBest XGBoost configuration:")
display(best_xgb_config)


Complete configurations: 8 / 8


,config_id,params_json,mean_mae,mean_rmse,mean_mape,mean_r2,std_rmse,worst_fold_rmse,min_fold_r2,folds_completed
0,xgb_config_01,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",834.812168,1111.552709,3.238868,0.931904,173.082926,1295.199684,0.907005,4
1,xgb_config_06,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",869.351335,1159.974736,3.362921,0.925602,183.896638,1382.626103,0.900944,4
2,xgb_config_04,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",905.261993,1216.410118,3.516314,0.917666,191.118822,1463.903960,0.888107,4
3,xgb_config_02,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",979.463769,1310.325225,3.799294,0.903740,192.604778,1559.289302,0.870171,4
4,xgb_config_05,"{""colsample_bytree"": 1.0, ""learning_rate"": 0.0...",1006.742195,1343.820956,3.936775,0.897348,173.973990,1571.631917,0.858815,4
5,xgb_config_08,"{""colsample_bytree"": 1.0, ""learning_rate"": 0.0...",1006.801852,1346.672858,3.940069,0.896757,176.036369,1581.882398,0.857766,4
6,xgb_config_03,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",1017.554618,1361.830144,3.956997,0.896632,226.472920,1670.516382,0.857185,4
7,xgb_config_07,"{""colsample_bytree"": 1.0, ""learning_rate"": 0.0...",1040.100330,1389.010820,4.064079,0.890258,183.414825,1636.272644,0.848229,4



Best XGBoost configuration:


{'model_type': 'XGBoost',
 'features': ['temperature_2m',
  'relative_humidity_2m',
  'dew_point_2m',
  'apparent_temperature',
  'precipitation',
  'rain',
  'surface_pressure',
  'cloud_cover',
  'wind_speed_10m',
  'wind_direction_10m',
  'shortwave_radiation',
  'econ_industrial_production_index_lag1m',
  'econ_gdp_index_lag1m',
  'econ_cpi_index_lag1m',
  'econ_unemployment_rate_lag1m',
  'econ_economic_data_complete',
  'hour',
  'day_of_week',
  'day_of_month',
  'month',
  'weekend',
  'is_holiday',
  'cal_event_count',
  'cal_is_covid_lockdown',
  'cal_is_general_election',
  'cal_is_major_football',
  'cal_is_non_working_day',
  'year',
  'day_of_year',
  'time_idx',
  'hour_sin',
  'hour_cos',
  'dow_sin',
  'dow_cos',
  'month_sin',
  'month_cos',
  'demand_lag_24',
  'demand_lag_168'],
 'params': {'colsample_bytree': 0.8,
  'learning_rate': 0.08,
  'max_depth': 6,
  'min_child_weight': 1,
  'n_estimators': 400,
  'reg_lambda': 2,
  'subsample': 1.0},
 'selection_metric': '

## 9. Compare XGBoost directly with the completed Prophet result

The Prophet values below come from the completed 4-fold tuning run. Its selected configuration was:

- `weather + lags + is_holiday`
- `changepoint_prior_scale = 0.05`
- `seasonality_prior_scale = 5.0`
- `seasonality_mode = multiplicative`

Primary comparison metric: **mean CV RMSE**.


In [10]:
# Completed Prophet result from the same four folds.
prophet_fold_metrics = pd.DataFrame([
    {
        "fold": "aug_2025",
        "mae": 1054.9259,
        "rmse": 1389.0176,
        "mape": 4.9326,
        "r2": 0.8075,
    },
    {
        "fold": "nov_2025",
        "mae": 1192.9037,
        "rmse": 1528.0188,
        "mape": 4.0235,
        "r2": 0.9353,
    },
    {
        "fold": "feb_2026",
        "mae": 1252.3714,
        "rmse": 1722.5748,
        "mape": 4.1182,
        "r2": 0.9072,
    },
    {
        "fold": "may_2026",
        "mae": 1054.7188,
        "rmse": 1395.9667,
        "mape": 4.8444,
        "r2": 0.8210,
    },
])

best_xgb_fold_metrics = (
    complete_folds[
        complete_folds["config_id"] == best_xgb_row["config_id"]
    ]
    [["fold", "mae", "rmse", "mape", "r2"]]
    .copy()
)

prophet_summary = {
    "model": "Prophet",
    "mean_mae": prophet_fold_metrics["mae"].mean(),
    "mean_rmse": prophet_fold_metrics["rmse"].mean(),
    "mean_mape": prophet_fold_metrics["mape"].mean(),
    "mean_r2": prophet_fold_metrics["r2"].mean(),
    "std_rmse": prophet_fold_metrics["rmse"].std(),
    "worst_fold_rmse": prophet_fold_metrics["rmse"].max(),
    "min_fold_r2": prophet_fold_metrics["r2"].min(),
}

xgb_summary = {
    "model": "XGBoost",
    "mean_mae": best_xgb_fold_metrics["mae"].mean(),
    "mean_rmse": best_xgb_fold_metrics["rmse"].mean(),
    "mean_mape": best_xgb_fold_metrics["mape"].mean(),
    "mean_r2": best_xgb_fold_metrics["r2"].mean(),
    "std_rmse": best_xgb_fold_metrics["rmse"].std(),
    "worst_fold_rmse": best_xgb_fold_metrics["rmse"].max(),
    "min_fold_r2": best_xgb_fold_metrics["r2"].min(),
}

model_comparison = (
    pd.DataFrame([prophet_summary, xgb_summary])
    .sort_values("mean_rmse")
    .reset_index(drop=True)
)

model_comparison.to_csv(
    OUTPUT_FOLDER / "prophet_vs_xgboost_cv.csv",
    index=False,
)

print("=== Prophet vs XGBoost — same folds ===")
display(model_comparison)

winner = model_comparison.iloc[0]["model"]

print(
    f"\nCV winner by mean RMSE: {winner}"
)
print(
    "\nIMPORTANT: June 2026 has still not been used."
)


=== Prophet vs XGBoost — same folds ===


,model,mean_mae,mean_rmse,mean_mape,mean_r2,std_rmse,worst_fold_rmse,min_fold_r2
0,XGBoost,834.812168,1111.552709,3.238868,0.931904,173.082926,1295.199684,0.907005
1,Prophet,1138.729950,1508.894475,4.479675,0.867750,156.149681,1722.574800,0.807500



CV winner by mean RMSE: XGBoost

IMPORTANT: June 2026 has still not been used.


## 10. Fold-by-fold comparison

In [11]:
fold_comparison = (
    prophet_fold_metrics
    .rename(
        columns={
            "mae": "prophet_mae",
            "rmse": "prophet_rmse",
            "mape": "prophet_mape",
            "r2": "prophet_r2",
        }
    )
    .merge(
        best_xgb_fold_metrics.rename(
            columns={
                "mae": "xgb_mae",
                "rmse": "xgb_rmse",
                "mape": "xgb_mape",
                "r2": "xgb_r2",
            }
        ),
        on="fold",
        how="inner",
    )
)

fold_comparison["rmse_winner"] = np.where(
    fold_comparison["xgb_rmse"] < fold_comparison["prophet_rmse"],
    "XGBoost",
    "Prophet",
)

fold_comparison.to_csv(
    OUTPUT_FOLDER / "prophet_vs_xgboost_by_fold.csv",
    index=False,
)

display(fold_comparison)


,fold,prophet_mae,prophet_rmse,prophet_mape,prophet_r2,xgb_mae,xgb_rmse,xgb_mape,xgb_r2,rmse_winner
0,aug_2025,1054.9259,1389.0176,4.9326,0.8075,684.149484,927.598554,3.185742,0.914142,XGBoost
1,nov_2025,1192.9037,1528.0188,4.0235,0.9353,935.068769,1217.318110,3.168528,0.958930,XGBoost
2,feb_2026,1252.3714,1722.5748,4.1182,0.9072,947.871594,1295.199684,3.084185,0.947539,XGBoost
3,may_2026,1054.7188,1395.9667,4.8444,0.8210,772.158826,1006.094490,3.517017,0.907005,XGBoost


---

# STOP HERE BEFORE FINAL JUNE TEST

At this point:

1. XGBoost has been tuned on the same four development folds as Prophet.
2. Both models have been compared using the same metrics and primary selection criterion.
3. June 2026 is still untouched.
4. Select the overall winner from the development CV comparison.
5. Only then run the final June test **once**.

The cell below is disabled by default.


## 11. Optional final June evaluation — enable only after overall model selection

In [12]:
RUN_FINAL_JUNE_TEST = False

if RUN_FINAL_JUNE_TEST:
    if winner != "XGBoost":
        print(
            "The CV winner is not XGBoost. "
            "Do not run the XGBoost June test unless XGBoost "
            "has been formally selected as the overall model."
        )
    else:
        # June contains lag_24/lag_168 values derived only from earlier observed demand.
        # This represents rolling daily next-24-hour operation.
        final_model_df = final_test_df.dropna(
            subset=feature_cols + ["demand_mw"]
        ).copy()

        X_train = dev_model_df[feature_cols]
        y_train = dev_model_df["demand_mw"]

        X_test = final_model_df[feature_cols]
        y_test = final_model_df["demand_mw"]

        final_model = xgb.XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            eval_metric="rmse",
            **best_xgb_params,
        )

        print(
            "Training XGBoost once on all development data "
            "through May 31, 2026..."
        )

        final_model.fit(
            X_train,
            y_train,
            eval_set=[(X_test, y_test)],
            verbose=False,
        )

        final_pred = final_model.predict(X_test)
        final_metrics = calculate_metrics(
            y_test,
            final_pred,
        )

        print("\n=== FINAL JUNE 2026 XGBoost RESULT ===")
        print(final_metrics)

        june_predictions = pd.DataFrame({
            "timestamp": final_model_df["timestamp"].values,
            "actual_demand_mw": y_test.values,
            "predicted_demand_mw": final_pred,
        })

        june_predictions.to_csv(
            OUTPUT_FOLDER / "xgb_final_june_predictions.csv",
            index=False,
        )

        with open(
            OUTPUT_FOLDER / "xgb_final_june_metrics.json",
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(final_metrics, f, indent=2)

        final_model.save_model(
            OUTPUT_FOLDER / "xgb_final_eval_model.json"
        )

        print(
            "\nSaved final June predictions, metrics, "
            "and the evaluation model."
        )
else:
    print(
        "RUN_FINAL_JUNE_TEST = False — June remains untouched."
    )


RUN_FINAL_JUNE_TEST = False — June remains untouched.


## 12. Optional production retrain

Run only after the final evaluation is finished and the selected model is approved for production. This retrains XGBoost once using all available historical data, including June.


In [13]:
RUN_PRODUCTION_RETRAIN = False

if RUN_PRODUCTION_RETRAIN:
    production_df = df.dropna(
        subset=feature_cols + ["demand_mw"]
    ).copy()

    production_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        eval_metric="rmse",
        **best_xgb_params,
    )

    production_model.fit(
        production_df[feature_cols],
        production_df["demand_mw"],
        verbose=False,
    )

    production_model.save_model(
        OUTPUT_FOLDER / "xgb_demand_model.json"
    )

    with open(
        OUTPUT_FOLDER / "xgb_feature_columns.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(feature_cols, f, indent=2)

    print(
        "Production model saved:",
        OUTPUT_FOLDER / "xgb_demand_model.json",
    )
else:
    print(
        "RUN_PRODUCTION_RETRAIN = False — "
        "production retrain not performed."
    )


RUN_PRODUCTION_RETRAIN = False — production retrain not performed.


## Expected output files

After the normal CV run (with both final-test flags left `False`), Kaggle should contain:

- `xgboost_outputs/xgb_tuning_folds_checkpoint.csv`
- `xgboost_outputs/xgb_tuning_folds.csv`
- `xgboost_outputs/xgb_tuning_summary.csv`
- `xgboost_outputs/best_xgb_config.json`
- `xgboost_outputs/prophet_vs_xgboost_cv.csv`
- `xgboost_outputs/prophet_vs_xgboost_by_fold.csv`

These are enough to choose between Prophet and XGBoost without touching June.
